# Synthetic Data Generation with SDV

[SDV](https://sdv.dev/) fits generative models to tabular, relational, and time-series data. This notebook uses **synthetic** source tables only and wraps optional dependencies in `try/except` so cells degrade gracefully when extras are not installed.

## Introduction: synthetic data and privacy

Synthetic records can stand in for production data in dev/test pipelines, balancing classes, or demos—**but** they are not automatically anonymous. Re-identification risk and statistical fidelity still need review (e.g., distance-to-closest-record checks, domain expert review). Treat SDV output as **useful, not guaranteed private** unless you validate for your threat model.

In [ ]:
try:
    import pandas as pd
    import numpy as np
    PANDAS_OK = True
except ImportError as e:
    PANDAS_OK = False
    print(e)

try:
    from sdv.single_table import GaussianCopulaSynthesizer
    from sdv.metadata import SingleTableMetadata
    SDV_SINGLE_OK = True
except ImportError as e:
    SDV_SINGLE_OK = False
    print("SDV single-table not available:", e)

PANDAS_OK, SDV_SINGLE_OK

## Single-table: `GaussianCopulaSynthesizer`

Models each column's distribution and copula-based dependencies—strong default for mixed numeric/categorical business tables.

In [ ]:
if PANDAS_OK and SDV_SINGLE_OK:
    rng = np.random.default_rng(7)
    n = 500
    df = pd.DataFrame({
        "region": rng.choice(["N", "S", "E", "W"], size=n),
        "revenue": rng.lognormal(3, 0.4, size=n).round(2),
        "active": rng.random(size=n) > 0.35,
    })
    meta = SingleTableMetadata()
    meta.detect_from_dataframe(df)
    gc = GaussianCopulaSynthesizer(meta)
    gc.fit(df)
    syn_gc = gc.sample(num_rows=50)
    print(syn_gc.head())
else:
    print("Skip GaussianCopula (pandas + sdv required).")

## Single-table: `CTGANSynthesizer` comparison

CTGAN uses a GAN-style generator; it can capture complex patterns but may need more rows and tuning than copula models.

In [ ]:
try:
    from sdv.single_table import CTGANSynthesizer
    CTGAN_OK = True
except ImportError as e:
    CTGAN_OK = False
    print("CTGAN not available:", e)

if PANDAS_OK and SDV_SINGLE_OK and CTGAN_OK:
    meta2 = SingleTableMetadata()
    meta2.detect_from_dataframe(df)
    ct = CTGANSynthesizer(meta2, epochs=20)  # small run for demo
    ct.fit(df)
    syn_ct = ct.sample(num_rows=50)
    print("CTGAN sample shape", syn_ct.shape)
    print("Real revenue mean", df["revenue"].mean(), "GC mean", syn_gc["revenue"].mean(), "CT mean", syn_ct["revenue"].mean())
else:
    print("Skip CTGAN comparison.")

## Relational data: `HMASynthesizer` concepts

**HMA** (Hierarchical Modeling Algorithm) synthesizes multi-table databases while respecting primary/foreign key relationships. You define **multi-table metadata**, fit on connected tables, then sample the whole schema.

In [ ]:
try:
    from sdv.metadata import MultiTableMetadata
    from sdv.multi_table import HMASynthesizer
    HMA_OK = True
except ImportError as e:
    HMA_OK = False
    print("HMA / multi-table SDV not available:", e)

if PANDAS_OK and HMA_OK:
    customers = pd.DataFrame({
        "customer_id": range(1, 201),
        "segment": np.random.choice(["SMB", "ENT"], 200),
    })
    orders = pd.DataFrame({
        "order_id": range(1, 401),
        "customer_id": np.random.randint(1, 201, 400),
        "amount": np.random.lognormal(2, 0.5, 400).round(2),
    })
    mmeta = MultiTableMetadata()
    mmeta.detect_table_from_dataframe("customers", customers)
    mmeta.detect_table_from_dataframe("orders", orders)
    mmeta.set_primary_key(table_name="customers", column_name="customer_id")
    mmeta.set_primary_key(table_name="orders", column_name="order_id")
    mmeta.add_relationship("customers", "orders", "customer_id", "customer_id")
    hma = HMASynthesizer(mmeta)
    hma.fit({"customers": customers, "orders": orders})
    syn_rel = hma.sample(scale=0.25)
    print({k: v.shape for k, v in syn_rel.items()})
else:
    print("Skip HMA demo.")

## Time-series: `PARSynthesizer` concepts

**PAR** models sequences per entity (e.g., per sensor or account) with a context column and time ordering. Typical steps: build **sequential** metadata, identify **entity** and **context** fields, fit, then sample series of similar length.

In [ ]:
try:
    from sdv.metadata import Metadata
    from sdv.sequential import PARSynthesizer
    PAR_OK = True
except ImportError as e:
    PAR_OK = False
    print("PAR / sequential SDV not available:", e)

if PANDAS_OK and PAR_OK:
    rows = []
    for entity in range(10):
        base = np.cumsum(np.random.randn(40)) + entity
        for t, v in enumerate(base):
            rows.append({"entity": str(entity), "time": t, "value": float(v)})
    ts_df = pd.DataFrame(rows)
    smeta = Metadata.detect_from_dataframe(ts_df, table_name="series")
    # SDV 1.x sequential API may vary; guard with try for column metadata updates
    try:
        smeta.update_column(table_name="series", column_name="entity", sdtype="id")
        smeta.set_sequence_key(table_name="series", column_name="entity")
        smeta.set_sequence_index(table_name="series", column_name="time")
        par = PARSynthesizer(smeta)
        par.fit(ts_df)
        syn_par = par.sample(num_sequences=3, sequence_length=15)
        print(syn_par.head(12))
    except Exception as ex:
        print("PAR demo skipped due to API/metadata issue:", ex)
else:
    print("Skip PAR demo.")

## Quality evaluation with SDMetrics

SDMetrics compares real vs. synthetic distributions (column shapes, correlations, multi-table integrity). Install `sdmetrics` alongside SDV when running full reports.

In [ ]:
try:
    from sdmetrics.reports.single_table import QualityReport
    SDMETRICS_OK = True
except ImportError as e:
    SDMETRICS_OK = False
    print("sdmetrics not available:", e)

if PANDAS_OK and SDV_SINGLE_OK and SDMETRICS_OK:
    report = QualityReport()
    report.generate(df, syn_gc, meta.to_dict())
    print(report.get_score())
    print(report.get_properties())
else:
    print("Skip QualityReport (needs sdmetrics + successful GaussianCopula run).")

## Privacy evaluation concepts

- **Membership inference**: can a classifier tell real vs. synthetic?
- **Attribute disclosure**: do synthetic rows reveal rare attribute combinations from the train set?
- **Distance to closest record (DCR)**: small distances suggest copying; compare to a null baseline.

SDV ecosystem includes privacy-oriented reports in some versions; always combine tooling with **policy** (who may access synthetic data, retention, and re-training cadence).

In [ ]:
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics.pairwise import euclidean_distances
    SK_OK = True
except ImportError:
    SK_OK = False

if PANDAS_OK and SK_OK and SDV_SINGLE_OK:
    num = df.select_dtypes(include=[np.number]).columns.tolist()
    if num:
        scaler = StandardScaler()
        R = scaler.fit_transform(df[num].values)
        S = scaler.transform(syn_gc[num].values)
        d = euclidean_distances(S, R).min(axis=1)
        print("Mean min distance real<-syn (numeric cols):", float(d.mean()))
    else:
        print("No numeric columns for distance demo.")
else:
    print("Skip DCR-style sketch.")

## Conclusion and use-case recommendations

| Use case | Often start with | Note |
|----------|------------------|------|
| Quick tabular prototype | `GaussianCopulaSynthesizer` | Fewer hyperparameters than deep models |
| Complex nonlinear columns | `CTGANSynthesizer` | Needs data volume; watch training time |
| Star/snowflake schemas | `HMASynthesizer` | Invest in correct keys and relationships |
| Event / panel sequences | `PARSynthesizer` | Define entity + sequence key carefully |

Install with your project's optional extra, e.g. `pip install 'agentic-assistants[synthetic-data]'`, validate with **SDMetrics**, and document **privacy assumptions** for any data that leaves trusted environments.